In [33]:
import os
import urllib.request
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [34]:
# Preparación de dependencias
os.system("pip uninstall -y torchao -q")
os.system("pip install -q peft")
from peft import LoraConfig, get_peft_model, TaskType

In [35]:
# Carga y correcciñon
data_file = None
for root, dirs, files in os.walk('/kaggle/input/'):
    for file in files:
        if file.endswith(('.tsv', '.csv')):
            data_file = os.path.join(root, file)
            break

if not data_file:
    data_file = "Restaurant_Reviews.tsv"
    if not os.path.exists(data_file):
        url = "https://raw.githubusercontent.com/zapata-diego/Restaurant-Reviews-Dataset/main/Restaurant_Reviews.tsv"
        urllib.request.urlretrieve(url, data_file)

df = pd.read_csv(data_file, sep='\t' if data_file.endswith('.tsv') else ',')

text_col = 'Review' if 'Review' in df.columns else df.columns[0]
rating_col = 'Liked' if 'Liked' in df.columns else df.columns[1]

df = df.dropna(subset=[text_col, rating_col]).copy()

# Correción
df['label'] = df[rating_col].astype(int)
df = df.rename(columns={text_col: 'text'})[['text', 'label']]

print("--> Distribución de clases:")
print(df['label'].value_counts().to_dict())

--> Distribución de clases:
{1: 500, 0: 500}


In [36]:
# Particiones y tokenizaciñon
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(test_df, test_size=0.50, random_state=42, stratify=test_df['label'])

raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df.reset_index(drop=True))
})

model_checkpoint = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128, padding="max_length")

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])

train_loader = DataLoader(tokenized_datasets["train"], batch_size=16, shuffle=True)
val_loader = DataLoader(tokenized_datasets["validation"], batch_size=16)
test_loader = DataLoader(tokenized_datasets["test"], batch_size=16)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n--> Ejecutando: {device}")

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]


--> Ejecutando: cuda


In [37]:
# Entrenamiento de BETO con LoRA

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=2 
)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

lora_model = get_peft_model(base_model, peft_config).to(device)

print("Parámetros entrenables con LORA")

lora_model.print_trainable_parameters()


optimizer = AdamW(lora_model.parameters(), lr=1e-3, weight_decay=0.01)

epochs = 3
print(f"--> Iniciando entrenamiento LoRA ({epochs} épocas)...")

for epoch in range(epochs):
    lora_model.train()
    total_train_loss = 0
    
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device).long()
        
        outputs = lora_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Evaluación conjunta de Validación
    lora_model.eval()
    val_preds, val_targets = [], []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device).long()
            
            outputs = lora_model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
            
    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds, average="weighted")
    print(f"Época {epoch + 1}/{epochs} | Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not 

Parámetros entrenables con LORA
trainable params: 296,450 || all params: 110,148,868 || trainable%: 0.2691
--> Iniciando entrenamiento LoRA (3 épocas)...
Época 1/3 | Loss: 0.6389 | Val Acc: 0.7200 | Val F1: 0.7182
Época 2/3 | Loss: 0.4661 | Val Acc: 0.7900 | Val F1: 0.7895
Época 3/3 | Loss: 0.3502 | Val Acc: 0.7700 | Val F1: 0.7689


In [38]:
# Evaluacion final de LoRA 

lora_model.eval()
test_preds, test_targets = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device).long()
        
        outputs = lora_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)
        
        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(labels.cpu().numpy())

lora_acc = accuracy_score(test_targets, test_preds)
lora_f1 = f1_score(test_targets, test_preds, average="weighted")


print(f"Resultados finales de LoRA")
print(f"BETO + LoRA -> Test Accuracy: {lora_acc:.4f} | Test F1-Puntuación: {lora_f1:.4f}")

Resultados finales de LoRA
BETO + LoRA -> Test Accuracy: 0.8900 | Test F1-Puntuación: 0.8899
